In [52]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/processed/modeling_datset.csv")

model_df = pd.read_csv(DATA_PATH)

train_df = model_df[model_df["issue_year"] <= 2015].copy()

validation_df = model_df[
    model_df["issue_year"].between(2016,2017)
].copy()

test_df = model_df[
    model_df["issue_year"] == 2018
].copy()

target_column = "default_flag"

feature_columns = [
    column for column in model_df.columns
    if column != target_column
]

x_train  = train_df[feature_columns]
y_train = train_df[target_column].astype("int8")

x_validation = validation_df[feature_columns]
y_validation = validation_df[target_column].astype("int8")

x_test = test_df[feature_columns]
y_test = test_df[target_column].astype("int8")

print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

Train: (829355, 13)
Validation: (462426, 13)
Test: (56318, 13)


In [53]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_features = x_train.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

categorical_features = x_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(strategy="most_frequent"),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=True,
                        ),
                    ),
                ]
            ),
            categorical_features,
        ),
    ]
)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("Tree preprocessor created successfully")

Numeric features: ['loan_amnt', 'term_months', 'int_rate', 'emp_length_years', 'annual_inc', 'issue_year', 'dti']
Categorical features: ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose']
Tree preprocessor created successfully


In [54]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "classifier",
            DecisionTreeClassifier(
                max_depth=8,
                min_samples_leaf=100,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ]
)

In [55]:
decision_tree_model.fit(x_train, y_train)

tree_predictions = decision_tree_model.predict(x_validation)
tree_probabilities = decision_tree_model.predict_proba(
    x_validation
)[:, 1]

print("Decision tree trained successfuly")
print("Predicted classes:", set(tree_predictions))

Decision tree trained successfuly
Predicted classes: {np.int8(0), np.int8(1)}


In [56]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", tree_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=50,
                max_depth=12,
                min_samples_leaf=100,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1
            ),
        ),
    ]
)

In [57]:
y_train = train_df[target_column].to_numpy().ravel()
y_validation = validation_df[target_column].to_numpy().ravel()
y_test = test_df[target_column].to_numpy().ravel()

In [58]:
random_forest_model.fit(x_train, y_train)

forest_predictions = random_forest_model.predict(x_validation)
forest_probabilities = random_forest_model.predict_proba(
    x_validation
)[:, 1]

print("Random Forest trained successfully.")
print("Predicted classes:", set(forest_predictions))

Random Forest trained successfully.
Predicted classes: {np.int64(0), np.int64(1)}


In [59]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

def evaluate_model(name, y_true, predictions, probabilities):
    return{
        "model": name,
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(
            y_true, predictions, zero_division=0
        ),
        "recall": recall_score(
            y_true, predictions, zero_division=0
        ),
        "f1": f1_score(
            y_true, predictions, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_true, probabilities),
    }

tree_results = pd.DataFrame([
    evaluate_model(
        "Decision tree",
        y_validation,
        tree_predictions,
        tree_probabilities,
    ),
    evaluate_model(
        "Random forest",
        y_validation,
        forest_predictions,
        forest_probabilities,
    ),
])

display(tree_results.round(4))

,model,accuracy,precision,recall,f1,roc_auc
0,Decision tree,0.6208,0.3405,0.6749,0.4526,0.6944
1,Random forest,0.6328,0.3482,0.6664,0.4574,0.6989


In [60]:
print("Feature columns:")
print(feature_columns)

print("\nTarget in feature columns:")
print(target_column in feature_columns)

print("\nX validation shape:", x_validation.shape)
print("\nY_validation shape:", y_validation.shape)

print("\nModeling dataset columns:")
print(model_df.columns.tolist())

print("\nPrediction agreement:")
print((forest_predictions == y_validation).mean())

print("\nConfusion Matrix:")
print(
    pd.crosstab(
        y_validation,
        forest_predictions,
        rownames=["actual"],
        colnames=["predicted"]
    )
)

Feature columns:
['loan_amnt', 'term_months', 'int_rate', 'grade', 'sub_grade', 'emp_length_years', 'home_ownership', 'annual_inc', 'verification_status', 'issue_year', 'purpose', 'dti']

Target in feature columns:
False

X validation shape: (462426, 12)

Y_validation shape: (462426,)

Modeling dataset columns:
['loan_amnt', 'term_months', 'int_rate', 'grade', 'sub_grade', 'emp_length_years', 'home_ownership', 'annual_inc', 'verification_status', 'issue_year', 'purpose', 'dti', 'default_flag']

Prediction agreement:
0.6327628636798104

Confusion Matrix:
predicted       0       1
actual                   
0          221025  133980
1           35840   71581


In [61]:
assert "default_flag" not in feature_columns
assert "outcome_group" not in feature_columns
assert "loan_status" not in feature_columns

print("No obvious target leakage in feature columns.")

No obvious target leakage in feature columns.


#### After correcting target leakage, model performance became realistic. Random forest currently provides the strongest balance, with an F1-score of 0.4574 and ROC-AUC of 0.6989. Logistic regression has higher recall, while the decision tree has the lowest overall balance.